# Module 2 — Practical Assignments: Embeddings, Chunking & Semantic Search

## Assignment 1 — Hands-on Practical (Semantic Search Walkthrough)

### Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu reportlab pypdf scikit-learn matplotlib

### Build a sample .txt file and a sample .pdf file to search over

In [ ]:
txt_source = (
    "Embeddings are dense numerical representations of text that capture semantic meaning. "
    "Unlike one-hot encoding or bag of words, embeddings place similar words and sentences "
    "close together in a continuous vector space, which allows machines to compare meaning "
    "rather than just exact word matches. Sentence Transformers are commonly used to generate "
    "these embeddings for search and retrieval tasks."
)

with open("doc1.txt", "w") as f:
    f.write(txt_source)

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

pdf_source = (
    "Chunking is the process of splitting large documents into smaller pieces before "
    "generating embeddings. Smaller chunks improve retrieval accuracy because the vector "
    "database can locate a precise, relevant piece of text instead of an entire lengthy "
    "document. Overlapping chunks are often used to prevent important context from being "
    "split across chunk boundaries."
)

pdf_styles = getSampleStyleSheet()
pdf_doc = SimpleDocTemplate("doc2.pdf", pagesize=letter)
pdf_doc.build([Paragraph(pdf_source, pdf_styles["Normal"])])

### Load both files back in

In [1]:
from pypdf import PdfReader

with open("doc1.txt", "r") as f:
    txt_content = f.read()

pdf_reader = PdfReader("doc2.pdf")
pdf_content = pdf_reader.pages[0].extract_text()

source_docs = [txt_content, pdf_content]
source_docs

['Embeddings are dense numerical representations of text that capture semantic meaning. Unlike one-hot encoding or bag of words, embeddings place similar words and sentences close together in a continuous vector space, which allows machines to compare meaning rather than just exact word matches. Sentence Transformers are commonly used to generate these embeddings for search and retrieval tasks.',
 'Chunking is the process of splitting large documents into smaller pieces before generating\nembeddings. Smaller chunks improve retrieval accuracy because the vector database can locate a\nprecise, relevant piece of text instead of an entire lengthy document. Overlapping chunks are often\nused to prevent important context from being split across chunk boundaries.\n']


### Split into chunks and embed them with a Sentence Transformer

In [1]:
from sentence_transformers import SentenceTransformer

text_chunks = []
for doc in source_docs:
    text_chunks.extend([piece.strip() for piece in doc.split(". ") if piece.strip()])

st_model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_embeddings = st_model.encode(text_chunks)
chunk_embeddings.shape

(6, 384)


### Compare two embedding models on the same query

In [1]:
from sklearn.metrics.pairwise import cosine_similarity

st_model_v2 = SentenceTransformer("all-mpnet-base-v2")
chunk_embeddings_v2 = st_model_v2.encode(text_chunks)

test_query = "What is chunking used for?"

query_vec_v1 = st_model.encode([test_query])
query_vec_v2 = st_model_v2.encode([test_query])

sims_v1 = cosine_similarity(query_vec_v1, chunk_embeddings)[0]
sims_v2 = cosine_similarity(query_vec_v2, chunk_embeddings_v2)[0]

print("all-MiniLM-L6-v2 similarities:", sims_v1)
print("all-mpnet-base-v2 similarities:", sims_v2)

all-MiniLM-L6-v2 similarities: [0.16963676 0.1345063  0.09531506 0.58399075 0.3548364  0.69957656]
all-mpnet-base-v2 similarities: [0.11477731 0.18702292 0.15697509 0.65311277 0.46487284 0.7040566 ]


### Build a FAISS index from the primary model's embeddings

In [1]:
import faiss
import numpy as np

vec_dim = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(vec_dim)
faiss_index.add(np.array(chunk_embeddings))
faiss_index.ntotal

6


### Run a semantic search against the index

In [1]:
query_vec = st_model.encode([test_query])
top_distances, top_indices = faiss_index.search(np.array(query_vec), 3)

for rank, idx in enumerate(top_indices[0]):
    print(f"Rank {rank + 1}: {text_chunks[idx]}  (distance: {top_distances[0][rank]:.4f})")

Rank 1: Overlapping chunks are often
used to prevent important context from being split across chunk boundaries.  (distance: 0.6008)
Rank 2: Chunking is the process of splitting large documents into smaller pieces before generating
embeddings  (distance: 0.8320)
Rank 3: Smaller chunks improve retrieval accuracy because the vector database can locate a
precise, relevant piece of text instead of an entire lengthy document  (distance: 1.2903)


### Plot the retrieved chunks by distance

In [ ]:
import matplotlib.pyplot as plt

top_chunks = [text_chunks[idx] for idx in top_indices[0]]
top_dists = top_distances[0]

plt.figure(figsize=(8, 4))
plt.barh(top_chunks, top_dists)
plt.xlabel("Distance (lower = more similar)")
plt.title("Semantic Search Results")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Assignment 2 — Mini Assignment: Semantic Document Search System

### Install dependencies

In [ ]:
!pip install -q sentence-transformers faiss-cpu

### Create three sample documents to search over

In [ ]:
corpus_texts = [
    "A vector database stores embeddings and enables fast similarity search instead of "
    "exact keyword matching. FAISS and ChromaDB are two popular choices for building "
    "retrieval systems.",
    "Cosine similarity measures the angle between two vectors rather than their magnitude, "
    "which makes it a common metric for comparing text embeddings of different lengths.",
    "Chunking strategies such as fixed size, recursive, semantic, and sliding window "
    "chunking all aim to split large documents into smaller pieces before embedding them.",
]

for i, text in enumerate(corpus_texts, start=1):
    with open(f"document_{i}.txt", "w") as f:
        f.write(text)

### Load the documents back from disk

In [1]:
corpus_docs = []
for i in range(1, 4):
    with open(f"document_{i}.txt", "r") as f:
        corpus_docs.append(f.read())

corpus_docs

['A vector database stores embeddings and enables fast similarity search instead of exact keyword matching. FAISS and ChromaDB are two popular choices for building retrieval systems.',
 'Cosine similarity measures the angle between two vectors rather than their magnitude, which makes it a common metric for comparing text embeddings of different lengths.',
 'Chunking strategies such as fixed size, recursive, semantic, and sliding window chunking all aim to split large documents into smaller pieces before embedding them.']


### Embed the documents

In [1]:
from sentence_transformers import SentenceTransformer

corpus_model = SentenceTransformer("all-MiniLM-L6-v2")
corpus_embeddings = corpus_model.encode(corpus_docs)
corpus_embeddings.shape

(3, 384)


### Index the embeddings with FAISS

In [1]:
import faiss
import numpy as np

vec_dim = corpus_embeddings.shape[1]
corpus_index = faiss.IndexFlatL2(vec_dim)
corpus_index.add(np.array(corpus_embeddings))
corpus_index.ntotal

3


### Take a query from the user and return the top matches

In [1]:
search_query = input("Enter your search query: ")

k = min(5, len(corpus_docs))
query_vec = corpus_model.encode([search_query])
result_distances, result_indices = corpus_index.search(np.array(query_vec), k)

for rank, idx in enumerate(result_indices[0]):
    print(f"Rank {rank + 1}: {corpus_docs[idx]}")
    print(f"Similarity Score (distance): {result_distances[0][rank]:.4f}")
    print("-" * 80)

Enter your search query: 3
Rank 1: Chunking strategies such as fixed size, recursive, semantic, and sliding window chunking all aim to split large documents into smaller pieces before embedding them.
Similarity Score (distance): 2.0225
--------------------------------------------------------------------------------
Rank 2: A vector database stores embeddings and enables fast similarity search instead of exact keyword matching. FAISS and ChromaDB are two popular choices for building retrieval systems.
Similarity Score (distance): 2.0531
--------------------------------------------------------------------------------
Rank 3: Cosine similarity measures the angle between two vectors rather than their magnitude, which makes it a common metric for comparing text embeddings of different lengths.
Similarity Score (distance): 2.1885
--------------------------------------------------------------------------------
